# Contact interfaces

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import *

lc = 0.5
c = Cylinder((0,0,0), X, 0.2, lc, bottom="inlet",
             top="interface", mantle="wall")
c.solids.name = "cyl"
b = Box((lc, -0.5, -0.5), (lc + 1, 0.5, 0.5))
b.faces.name = "wall"
b.faces.Max(X).name = "outlet"
b = Glue([b, c.faces["interface"][0]]).Move((0.2, 0, 0))
b.faces.maxh=0.08
b.solids.name = "box"
b.faces["interface"].name = "interfaceb"

geo = OCCGeometry(Compound([c, b]))
# Draw(geo)

mesh = Mesh(geo.GenerateMesh(maxh=0.1))
mesh.Curve(3)
Draw(mesh)
print(set(mesh.GetBoundaries()), mesh.GetMaterials())

In [ ]:
order = 3
fes = Discontinuous(HDiv(mesh, order=order)) * L2(mesh, order=order-1) * FacetFESpace(mesh, order=order, dirichlet="inlet|outlet")
(sigma, u, uhat), (tau, v, vhat) = fes.TnT()

n = specialcf.normal(mesh.dim)
dS = dx(element_boundary=True)

## Mortar method with hybrid-mixed:

In [ ]:
mass2 = BilinearForm(uhat*vhat.Dual()*ds("interfaceb"), diagonal=True).Assemble()

massmixed = BilinearForm(fes)
massmixed += ContactIntegrator(me="interfaceb", other="interface", intorder=50).Add(uhat.Other()*vhat.Dual())
massmixed.Assemble();

In [ ]:
gfu = GridFunction(fes)
gfu.components[2].Set ( mesh.BoundaryCF( { "interface" : y }, default=0), definedon=mesh.Boundaries(".*") )
Draw (gfu.components[2].Trace(), mesh, draw_vol=False);

In [ ]:
print (mesh.Boundaries("interfaceb").Mask())
secdofs = fes.GetDofs(mesh.Boundaries("interfaceb"))
print (secdofs.NumSet())

Primary-secondary projector can be built as `SparseMatrix`:

$$
P_{1\rightarrow2} = M_2^{-1} M_{21}
$$

For the facet spaces, $M_2$ is block-diagonal.

Projector: For a given function, projector takes the primary function to the secondary side.

In [ ]:
projectorPS = (mass2.mat.Inverse(secdofs) @ massmixed.mat.DeleteZeroElements(1e-12)).CreateSparseMatrix()
projector = (Projector(secdofs, False) + projectorPS).CreateSparseMatrix()
print (projectorPS.nze, projector.nze)

In [ ]:
gfu2 = GridFunction(fes)
gfu2.vec.data = projector*gfu.vec

In [ ]:
Draw (gfu2.components[2].Trace(), mesh, draw_vol=False);

In [ ]:
a = BilinearForm(fes, condense=True)
a += (sigma * tau + grad(u) * tau + sigma * grad(v)) * dx
dS = dx(element_boundary=True)
a += -(sigma * n * (v-vhat) + (u-uhat) * n * tau) * dS
a.Assemble()

f = LinearForm(fes)

In [ ]:
a.mat.nze, projector.nze, projectorPS.nze

In [ ]:
# invS = a.mat.Inverse(freedofs=fes.FreeDofs(coupling=True))
with TaskManager():
    projectedS = (projector.T@a.mat@projector).CreateSparseMatrix()
    invS = projector@projectedS.Inverse(inverse="sparsecholesky", freedofs=fes.FreeDofs(coupling=True) & ~secdofs)@projector.T
    ext = IdentityMatrix() + a.harmonic_extension
    extT = IdentityMatrix() + a.harmonic_extension_trans
    invA =  ext @ invS @ extT + a.inner_solve

In [ ]:
gf = GridFunction(fes)
gf.components[2].Interpolate(1+y, definedon=mesh.Boundaries("inlet"))
gf.vec.data += a.harmonic_extension * gf.vec
gf.vec.data += invA * (f.vec - a.mat * gf.vec)

In [ ]:
gfsigma, gfu, gfuhat = gf.components
Draw (gfuhat, draw_vol=False);

In [ ]:
Draw (gfu, draw_surf=False, clipping={ "function" : True, "vec":(0,0,-1) });

In [ ]:
Draw (gfsigma, mesh, draw_surf=False, clipping={ "function" : True, "vec":(0,0,-1) });